In [2]:
import mne

from mne.datasets.sleep_physionet.age import fetch_data

print("MNE version:", mne.__version__)

MNE version: 1.12.1


In [3]:
files = fetch_data(
    subjects=[0],
    recording=[1]
)

print(files)

Using default location ~/mne_data for PHYSIONET_SLEEP...
[['C:\\Users\\user\\mne_data\\physionet-sleep-data\\SC4001E0-PSG.edf', 'C:\\Users\\user\\mne_data\\physionet-sleep-data\\SC4001EC-Hypnogram.edf']]


C:\Users\user\anaconda3\envs\neuro_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
psg_file, hypnogram_file = files[0]

raw = mne.io.read_raw_edf(
    psg_file,
    preload=True
)

annotations = mne.read_annotations(
    hypnogram_file
)

raw.set_annotations(
    annotations
)

print(raw)
print(raw.ch_names)

Extracting EDF parameters from C:\Users\user\mne_data\physionet-sleep-data\SC4001E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 7949999  =      0.000 ... 79499.990 secs...


C:\Users\user\AppData\Local\Temp\ipykernel_2384\2313655372.py:3: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(
C:\Users\user\AppData\Local\Temp\ipykernel_2384\2313655372.py:3: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(
C:\Users\user\AppData\Local\Temp\ipykernel_2384\2313655372.py:3: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(


<RawEDF | SC4001E0-PSG.edf, 7 x 7950000 (79500.0 s), ~424.6 MiB, data loaded>
['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']


C:\Users\user\AppData\Local\Temp\ipykernel_2384\2313655372.py:12: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(


In [5]:
print(raw.annotations)

print("\nПерші 20 annotation:")
for ann in raw.annotations[:20]:
    print(ann)

<Annotations | 154 segments: Sleep stage 1 (24), Sleep stage 2 (40), Sleep ...>

Перші 20 annotation:
OrderedDict([('onset', np.float64(0.0)), ('duration', np.float64(30630.0)), ('description', np.str_('Sleep stage W')), ('orig_time', datetime.datetime(1989, 4, 24, 16, 13, tzinfo=datetime.timezone.utc)), ('extras', {})])
OrderedDict([('onset', np.float64(30630.0)), ('duration', np.float64(120.0)), ('description', np.str_('Sleep stage 1')), ('orig_time', datetime.datetime(1989, 4, 24, 16, 13, tzinfo=datetime.timezone.utc)), ('extras', {})])
OrderedDict([('onset', np.float64(30750.0)), ('duration', np.float64(390.0)), ('description', np.str_('Sleep stage 2')), ('orig_time', datetime.datetime(1989, 4, 24, 16, 13, tzinfo=datetime.timezone.utc)), ('extras', {})])
OrderedDict([('onset', np.float64(31140.0)), ('duration', np.float64(30.0)), ('description', np.str_('Sleep stage 3')), ('orig_time', datetime.datetime(1989, 4, 24, 16, 13, tzinfo=datetime.timezone.utc)), ('extras', {})])
OrderedDi

In [6]:
annotation_desc_2_event_id = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3": 4,
    "Sleep stage 4": 4,
    "Sleep stage R": 5,
}

events, event_id = mne.events_from_annotations(
    raw,
    event_id=annotation_desc_2_event_id,
    chunk_duration=30.0
)

print("EVENT ID:")
print(event_id)

print("\nEVENTS SHAPE:")
print(events.shape)

print("\nFIRST 10 EVENTS:")
print(events[:10])

Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
EVENT ID:
{np.str_('Sleep stage 1'): 2, np.str_('Sleep stage 2'): 3, np.str_('Sleep stage 3'): 4, np.str_('Sleep stage 4'): 4, np.str_('Sleep stage R'): 5, np.str_('Sleep stage W'): 1}

EVENTS SHAPE:
(2650, 3)

FIRST 10 EVENTS:
[[    0     0     1]
 [ 3000     0     1]
 [ 6000     0     1]
 [ 9000     0     1]
 [12000     0     1]
 [15000     0     1]
 [18000     0     1]
 [21000     0     1]
 [24000     0     1]
 [27000     0     1]]


In [7]:
epochs = mne.Epochs(
    raw,
    events,
    event_id=event_id,
    tmin=0,
    tmax=30 - 1 / raw.info["sfreq"],
    baseline=None,
    preload=True
)

print(epochs)

Not setting metadata
2650 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2650 events and 3000 original time points ...
0 bad epochs dropped
<Epochs | 2650 events (all good), 0 – 29.99 s (baseline off), ~424.6 MiB, data loaded,
 np.str_('Sleep stage 1'): 58
 np.str_('Sleep stage 2'): 250
 np.str_('Sleep stage 3'): 220
 np.str_('Sleep stage 4'): 220
 np.str_('Sleep stage R'): 125
 np.str_('Sleep stage W'): 1997>


In [8]:
X = epochs.get_data()
y = epochs.events[:, 2]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("First 20 labels:")
print(y[:20])

X shape: (2650, 7, 3000)
y shape: (2650,)
First 20 labels:
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [9]:
eeg_picks = mne.pick_types(
    epochs.info,
    eeg=True,
    eog=False,
    emg=False,
    misc=False
)

X_eeg = epochs.get_data(picks=eeg_picks)

print("EEG channels:", [epochs.ch_names[i] for i in eeg_picks])
print("X_eeg shape:", X_eeg.shape)

EEG channels: ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']
X_eeg shape: (2650, 7, 3000)


In [10]:
raw.set_channel_types({
    "EEG Fpz-Cz": "eeg",
    "EEG Pz-Oz": "eeg",
    "EOG horizontal": "eog",
    "Resp oro-nasal": "misc",
    "EMG submental": "emg",
    "Temp rectal": "misc",
    "Event marker": "misc",
})

C:\Users\user\AppData\Local\Temp\ipykernel_2384\1633227951.py:1: RuntimeWarning: The unit for channel(s) Event marker, Resp oro-nasal, Temp rectal has changed from V to NA.
  raw.set_channel_types({


<RawEDF | SC4001E0-PSG.edf, 7 x 7950000 (79500.0 s), ~424.6 MiB, data loaded>

In [11]:
epochs = mne.Epochs(
    raw,
    events,
    event_id=event_id,
    tmin=0,
    tmax=30 - 1 / raw.info["sfreq"],
    baseline=None,
    preload=True
)

Not setting metadata
2650 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2650 events and 3000 original time points ...
0 bad epochs dropped


In [12]:
eeg_picks = mne.pick_types(
    epochs.info,
    eeg=True,
    eog=False,
    emg=False,
    misc=False
)

X_eeg = epochs.get_data(picks=eeg_picks)

print("EEG channels:", [epochs.ch_names[i] for i in eeg_picks])
print("X_eeg shape:", X_eeg.shape)

EEG channels: ['EEG Fpz-Cz', 'EEG Pz-Oz']
X_eeg shape: (2650, 2, 3000)


In [13]:
import numpy as np

labels, counts = np.unique(y, return_counts=True)

print("Labels:", labels)
print("Counts:", counts)

Labels: [1 2 3 4 5]
Counts: [1997   58  250  220  125]


In [14]:
sleep_indices = np.where(y != 1)[0]

first_sleep = sleep_indices[0]
last_sleep = sleep_indices[-1]

margin = 60   # 60 епох × 30 секунд = 30 хвилин

start = max(0, first_sleep - margin)
stop = min(len(y), last_sleep + margin + 1)

X_trim = X_eeg[start:stop]
y_trim = y[start:stop]

print("Original:", X_eeg.shape, y.shape)
print("Trimmed:", X_trim.shape, y_trim.shape)

labels, counts = np.unique(y_trim, return_counts=True)

print("Labels:", labels)
print("Counts:", counts)

Original: (2650, 2, 3000) (2650,)
Trimmed: (841, 2, 3000) (841,)
Labels: [1 2 3 4 5]
Counts: [188  58 250 220 125]


In [15]:
label_map = {
    1: 0,
    2: 1,
    3: 2,
    4: 3,
    5: 4
}

y_ml = np.array([
    label_map[label]
    for label in y_trim
])

print("Old labels:", np.unique(y_trim))
print("New labels:", np.unique(y_ml))

print("First 20 new labels:")
print(y_ml[:20])

Old labels: [1 2 3 4 5]
New labels: [0 1 2 3 4]
First 20 new labels:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [16]:
X_mean = X_trim.mean(
    axis=2,
    keepdims=True
)

X_std = X_trim.std(
    axis=2,
    keepdims=True
)

X_norm = (
    X_trim - X_mean
) / (X_std + 1e-8)

print("X_norm shape:", X_norm.shape)

print("Mean first epoch:")
print(X_norm[0].mean())

print("Std first epoch:")
print(X_norm[0].std())

X_norm shape: (841, 2, 3000)
Mean first epoch:
-4.1448326252672514e-18
Std first epoch:
0.9988711234505266


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_norm,
    y_ml,
    test_size=0.30,
    random_state=42,
    stratify=y_ml
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (588, 2, 3000) (588,)
Validation: (126, 2, 3000) (126,)
Test: (127, 2, 3000) (127,)


In [18]:
import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

2.14.0+cpu
CUDA available: False


In [19]:
import torch
from torch.utils.data import Dataset


class EEGSleepDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )


    def __len__(self):
        return len(self.y)


    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [20]:
train_dataset = EEGSleepDataset(
    X_train,
    y_train
)

val_dataset = EEGSleepDataset(
    X_val,
    y_val
)

test_dataset = EEGSleepDataset(
    X_test,
    y_test
)

In [21]:
print("Train dataset length:", len(train_dataset))

x0, y0 = train_dataset[0]

print("First EEG shape:", x0.shape)
print("First label:", y0)
print("EEG dtype:", x0.dtype)
print("Label dtype:", y0.dtype)

Train dataset length: 588
First EEG shape: torch.Size([2, 3000])
First label: tensor(0)
EEG dtype: torch.float32
Label dtype: torch.int64


In [22]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

X_batch, y_batch = next(iter(train_loader))

print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)

X batch shape: torch.Size([32, 2, 3000])
y batch shape: torch.Size([32])


In [23]:
import torch
import torch.nn as nn


class EEGCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=7,
            padding=3
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(
            kernel_size=2
        )

        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            padding=2
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(
            32,
            5
        )


    def forward(self, x):

        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.relu(x)

        x = self.global_pool(x)

        x = x.squeeze(-1)

        x = self.fc(x)

        return x

In [24]:
model = EEGCNN()

print(model)

EEGCNN(
  (conv1): Conv1d(2, 16, kernel_size=(7,), stride=(1,), padding=(3,))
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
  (global_pool): AdaptiveAvgPool1d(output_size=1)
  (fc): Linear(in_features=32, out_features=5, bias=True)
)


In [25]:
X_batch, y_batch = next(iter(train_loader))

output = model(X_batch)

print("Input shape:", X_batch.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([32, 2, 3000])
Output shape: torch.Size([32, 5])


In [26]:
import numpy as np
import torch
import torch.nn as nn


# Створюємо нову модель
model = EEGCNN()


# Рахуємо, скільки прикладів кожного класу в train
class_counts = np.bincount(
    y_train,
    minlength=5
)

print("Class counts:", class_counts)


# Робимо ваги класів
class_weights = len(y_train) / (
    5 * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("Class weights:", class_weights)


# Функція помилки
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# Оптимізатор
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

Class counts: [131  41 175 154  87]
Class weights: tensor([0.8977, 2.8683, 0.6720, 0.7636, 1.3517])


In [27]:
epochs_count = 10


for epoch in range(epochs_count):

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0


    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        optimizer.step()


        train_loss += loss.item()

        predictions = outputs.argmax(
            dim=1
        )

        train_correct += (
            predictions == y_batch
        ).sum().item()

        train_total += y_batch.size(0)


    train_accuracy = (
        train_correct
        / train_total
        * 100
    )


    model.eval()

    val_correct = 0
    val_total = 0


    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            outputs = model(X_batch)

            predictions = outputs.argmax(
                dim=1
            )

            val_correct += (
                predictions == y_batch
            ).sum().item()

            val_total += y_batch.size(0)


    val_accuracy = (
        val_correct
        / val_total
        * 100
    )


    print(
        f"Epoch {epoch + 1}/{epochs_count} | "
        f"Loss: {train_loss:.4f} | "
        f"Train accuracy: {train_accuracy:.1f}% | "
        f"Validation accuracy: {val_accuracy:.1f}%"
    )

Epoch 1/10 | Loss: 30.8227 | Train accuracy: 14.8% | Validation accuracy: 15.1%
Epoch 2/10 | Loss: 30.3589 | Train accuracy: 24.3% | Validation accuracy: 21.4%
Epoch 3/10 | Loss: 30.0591 | Train accuracy: 27.0% | Validation accuracy: 38.9%
Epoch 4/10 | Loss: 29.5280 | Train accuracy: 49.0% | Validation accuracy: 60.3%
Epoch 5/10 | Loss: 28.7977 | Train accuracy: 47.8% | Validation accuracy: 61.1%
Epoch 6/10 | Loss: 27.5112 | Train accuracy: 55.3% | Validation accuracy: 53.2%
Epoch 7/10 | Loss: 26.0550 | Train accuracy: 58.3% | Validation accuracy: 55.6%
Epoch 8/10 | Loss: 24.4829 | Train accuracy: 54.1% | Validation accuracy: 64.3%
Epoch 9/10 | Loss: 22.9356 | Train accuracy: 65.3% | Validation accuracy: 62.7%
Epoch 10/10 | Loss: 21.9751 | Train accuracy: 66.3% | Validation accuracy: 70.6%


In [28]:
model.eval()

test_correct = 0
test_total = 0

all_predictions = []
all_labels = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)

        predictions = outputs.argmax(dim=1)

        test_correct += (
            predictions == y_batch
        ).sum().item()

        test_total += y_batch.size(0)

        all_predictions.extend(
            predictions.numpy()
        )

        all_labels.extend(
            y_batch.numpy()
        )


test_accuracy = (
    test_correct / test_total * 100
)

print(
    f"Test accuracy: {test_accuracy:.1f}%"
)

Test accuracy: 70.9%


In [29]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

print()

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Wake",
            "N1",
            "N2",
            "N3",
            "REM"
        ],
        zero_division=0
    )
)

[[26  0  2  1  0]
 [ 6  1  0  0  1]
 [ 0  0 30  4  4]
 [ 0  0  3 30  0]
 [ 6  9  1  0  3]]

              precision    recall  f1-score   support

        Wake       0.68      0.90      0.78        29
          N1       0.10      0.12      0.11         8
          N2       0.83      0.79      0.81        38
          N3       0.86      0.91      0.88        33
         REM       0.38      0.16      0.22        19

    accuracy                           0.71       127
   macro avg       0.57      0.58      0.56       127
weighted avg       0.69      0.71      0.69       127

